# Prompt Anatomy and Secure Model Connectivity

### What is an LLM


![](./images/llm_architecture_basic.png)

**Basic LLM Architecture**


#### LLM Model stats:

- What are parameters of an LLM? 8B, 100B etc..

- (Tokens per Second) TPS means? 

#### **Prompt:**

- Natural language and context payload supplied to LLM to steer its auto generative token generation

### Core Anatomy of an Enterprise Prompt

1. Instruction: The primary imperative task directive telling the model what action to execute (e.g., "Extract entities", "Summarize text").  

2. Context / Persona: Background information, operational boundaries, or behavioral framing (e.g., "You are an enterprise data engineer").  

3. Input Data: The variable payload or raw source text to be processed, typically isolated within explicit delimiters (""" or ###).  

4. Output Indicator: Formatting constraints and structural targets specifying the exact response schema (e.g., valid JSON schema, Markdown table, or bullet points).

### The Master Prompt Template (Open AI)
leverages structured formatting like XML tags to separate the role, context, task, and formatting rules

### SYSTEM PERSONA
You are an enterprise AI data ingestion engine specializing in customer support automation. 
Your job is to extract triage metadata from incoming customer tickets with high deterministic precision. 
Adhere strictly to the provided output schema and never invent or infer data not grounded in the source text.

### INSTRUCTION
1. Analyze the customer support ticket provided in the payload.
2. Identify and extract:
   - Primary customer issue
   - Urgency level (`LOW`, `MEDIUM`, `HIGH`, `CRITICAL`)
   - Sentiment score (`POSITIVE`, `NEUTRAL`, `NEGATIVE`)
   - Mentioned products/services
   - Action items for the support team
3. If information for a field is missing, set its value to `null`.
4. Suppress all conversational preamble, explanations, and markdown commentary outside the requested schema.

### INPUT DATA
"""
Ticket ID: #INC-94821
User: devops-lead@enterprise-client.com
Timestamp: 2026-08-23T10:14:00Z
Message:
Our production cluster on AWS us-east-1 went down 15 minutes ago after we pushed the latest database migration. The pgvector extension is throwing memory allocation errors, causing all semantic search queries to fail with HTTP 500 status codes. This is blocking our core checkout service. We need immediate assistance from the database infrastructure team to rollback or resize the instance.
"""

### OUTPUT INDICATOR
Respond strictly with valid JSON conforming to the following structure:
{
  "ticket_id": "string",
  "urgency": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",
  "sentiment": "POSITIVE" | "NEUTRAL" | "NEGATIVE",
  "issue_summary": "string",
  "affected_components": ["string"],
  "action_items": ["string"]
}

### Google GenAI Python SDK

In [ ]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("Gemini API KEY not found!")

In [ ]:
client = genai.Client(api_key=gemini_api_key)
prompt = "Tell me a joke on AI Engineering"

response = client.models.generate_content(
    model = "gemini-3.7-flash",
    contents = prompt
)


In [ ]:
print(response.text)

### Primary API call Methods

- Standard Invocation: client.models.generate_content(model="gemini-2.5-flash", contents=...)  

- Streaming Protocol: client.models.generate_content_stream(model="gemini-2.5-flash", contents=...) 

- Async Invocation (FastAPI Integration): await client.aio.models.generate_content(...)  

- Vector Embeddings (RAG / Session 20): client.models.embed_content(model="text-embedding-004", contents=...) 

** Model Configurations: Passed via google.genai.types.GenerateContentConfig (handles system_instruction, temperature, max_output_tokens, and response_schema)

1. Standard Synchronous Call (Blocking)

Architecture: Blocks the executing Python thread until the model generates the entire response.  

Mechanism: The client sends an HTTP POST request to the inference endpoint and deserializes the full JSON payload upon completion.

When to Use: Batch scripts, data transformation jobs, or headless backend tasks where real-time user feedback is unnecessary.

In [7]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents="Explain vector embeddings in detail",
    config=types.GenerateContentConfig(
        temperature=0.2,
        max_output_tokens=150,
    ),
)

# Access generated text directly
print(response.text)
with open("output.txt","w") as file:
    file.write(response.text)

Numbers vs. Human Concepts


2. Streaming Call (generate_content_stream)

Architecture: Establishes a persistent Server-Sent Events (SSE) connection.  

Mechanism: Instead of waiting for 500+ tokens to compute, the server yields chunks as soon as attention decode layers finish sampling them. This drastically cuts Time-to-First-Token (TTFT) from ~1.5s down to ~150ms.  

When to Use: Interactive user interfaces/ Dashborads and real-time terminal outputs.

In [5]:
from google import genai

client = genai.Client()

response_stream = client.models.generate_content_stream(
    model="gemini-2.5-flash",
    contents="Explain what is the Model Context Protocol (MCP) in detail.",
)

for chunk in response_stream:
    if chunk.text:
        # flush=True forces immediate output to console without buffer delay
        print(chunk.text, end="", flush=True)
#print()

The **Model Context Protocol (MCP)** is not a single, universally standardized or officially ratified protocol like HTTP or TCP/IP. Instead, it's an *umbrella term* or a *conceptual framework* that refers to the **set of strategies, rules, and mechanisms designed to manage, structure, and provide relevant contextual information to a large language model (LLM) or other AI models**, especially to overcome their inherent limitations regarding context window size and long-term memory.

In essence, an MCP defines *how* an application or system interacts with an LLM to ensure the model always has the most pertinent information at its disposal, regardless of the length or complexity of the ongoing interaction or the external knowledge required.

Let's break down the Model Context Protocol in detail.

---

### The Core Problem MCP Addresses: LLM Context Window Limitations

Large Language Models, particularly transformer-based models, operate with a fixed "context window" (also known as token w

3. Asynchronous Invocation (client.aio)

Architecture: Integrates with Python’s asyncio event loop without blocking the main worker thread.  

Mechanism: Yields execution during the network I/O wait, allowing a single server worker to handle hundreds of concurrent requests simultaneously.  

When to Use: FastAPI Microservices, cyclic state machines (LangGraph), and concurrent web scrapers.

In [ ]:
import asyncio
from google import genai

client = genai.Client()

async def generate_summary(prompt: str) -> str:
    response = await client.aio.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

async def main():
    tasks = [
        generate_summary("Define Prompt Engineering in 10 words."),
        generate_summary("Define Retrieval-Augmented Generation in 10 words."),
    ]
    # Execute both API calls in parallel
    results = await asyncio.gather(*tasks)
    for res in results:
        print("->", res)

asyncio.run(main())

4. Structured Schema Enforcement (response_schema & Pydantic)

Architecture: Constrains the sampling logits at the decode layer to strictly conform to a JSON grammar/schema.

Mechanism: Guarantees that the output parses directly into a Pydantic object without requiring brittle regex or JSON-repair fallbacks.  

When to Use: Data extraction pipelines, database ingestion, and agent tool parameter extraction. 

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

class Incident(BaseModel):
    urgency: str = Field(description="LOW, MEDIUM, HIGH, or CRITICAL")
    affected_service: str
    action_items: list[str]

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="PostgreSQL database running out of connections on us-east-1 production node.",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Incident,
        temperature=0.0,
    ),
)

print(response.text)  # Guaranteed valid JSON conforming strictly to IncidentTriage

In [ ]:
from google import genai
from google.genai import types
from pydantic import BaseModel, Field


# 1. Define the Pydantic schema for the book details
class Book(BaseModel):
    title: str = Field(description="Exact title of the book")
    author: str = Field(description="Full name of the author(s)")
    edition: str = Field(
        description="Latest edition information (e.g., '3rd Edition', '2nd Edition', or '1st Edition')"
    )
    publisher: str = Field(description="Publishing company or academic press")
    publication_year: int = Field(description="Year of publication for this edition")

class BookCatalog(BaseModel):
    subject: str = Field(description="The user requested subject or domain")
    books: list[Book] = Field(description="List of recommended and verified books")

client = genai.Client()
# subject = "Engineering Mathematics"
# count = 3
def get_textbooks(count,subject):
    prompt = f"""
    Search the web for {count} of the most authoritative and recommended textbooks/reference books on the subject: "{subject}".
    Look up real, verified details for each book including the latest edition, publisher, publication year, and topics.
    """

    response = client.models.generate_content(
        model="gemini-3.7-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            # Enable Google Search grounding
            tools=[types.Tool(google_search=types.GoogleSearch())],
            # Enforce strict JSON conforming to the Pydantic model
            response_mime_type="application/json",
            response_schema=BookCatalog,
            temperature=0.0,
        ),
    )
    return(response.text)

if __name__ == "__main__":
    books = get_textbooks(count=3,subject="Engineering Mathematics")
    print(books)

In [ ]:
print(response.text)

5. Dense Vector Embeddings (embed_content)

Architecture: Queries the text representation layer to produce fixed-dimension dense vectors (D=768).  

Mechanism: Transforms raw semantic text into vector space coordinates used for nearest-neighbor search (Cosine/Dot Product) in Phase 4 (RAG).  

When to Use: Ingesting documents into ChromaDB, Pinecone, or PostgreSQL (pgvector) (Sessions 19–21).

In [4]:
from google import genai

client = genai.Client()

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents="Hybrid search pairs BM25 keyword matching with dense embeddings.",
)

# Extract embedding float array
vector = result.embeddings[0].values
print(f"Embedding Vector Dimension: {len(vector)}")  # 768
print(f"Sample Vector Values: {vector[:3]}...")

Embedding Vector Dimension: 3072
Sample Vector Values: [-0.014123147, 0.017321605, -0.0206569]...


In [ ]:
from google import genai

client = genai.Client()

keyword = "pro"  # e.g., "pro", "flash", "embedding", "veo"

filtered_models = [
    model.name.replace("models/", "")
    for model in client.models.list()
    if keyword.lower() in model.name.lower()
]

print(f"Models matching '{keyword}':")
for name in filtered_models:
    print(f"- {name}")

6. Interactions API (Recommended/latest)

- Google’s unified interface across the google-genai SDK for interacting with both foundation models and autonomous agents (such as Deep Research).
- Interactions API introduces server-side state management.
- Conversation Forking: Branch a discussion into multiple paths from any historical interaction.id without state mutation.
- Native Multimodal Generation: Supports text, image, and audio outputs within the same execution flow.

## Case Study: Stateful LLM Chat

**Scenario:** You got a SMS stating your credit card was debited with Rs. 10,000 without your knowledge. Get help from gemini falsh LLM to Investigate the transaction, without sending same prompt texts to and fro.

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

# 1. Load API Key and initialize client
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 2. Turn 1: Initial user symptom report
print("--- [Turn 1: User Reports Issue] ---")
turn_1 = client.interactions.create(
    model="gemini-2.5-flash",
    input="Hi, I got an SMS stating that Rs.10,000 was debited from my credit card without my consent."
)

# Extract generated response text
print(f"Assistant: {turn_1.outputs[-1].text}\n")
print(f"Interaction ID: {turn_1.id}")

# 3. Turn 2: Follow-up question referencing only previous_interaction_id
print("\n--- [Turn 2: Asking for Remediation without re-sending history] ---")
turn_2 = client.interactions.create(
    model="gemini-2.5-flash",
    input="What should I do to make sure my credit card will not be misused?",
    previous_interaction_id=turn_1.id  # Server automatically injects Turn 1 context
)

print(f"Assistant: {turn_2.outputs[-1].text}\n")

# 4. Turn 3 (Forking/Branching): Asking an alternative question from Turn 1
print("\n--- [Turn 3: Branching/Forking Context from Turn 1] ---")
turn_3_fork = client.interactions.create(
    model="gemini-2.5-flash",
    input="May be my wife used it for shopping, how to know without hurting her feelings",
    previous_interaction_id=turn_1.id  # Directly branches from Turn 1
)

print(f"Assistant (Forked Branch): {turn_3_fork.outputs[-1].text}")